# Top-20 LightGBM Evaluation

In [ ]:
from pathlib import Path

import lightgbm as lgb
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_DIR = PROJECT_ROOT / "data" / "top20_p8"
MODEL_PATH = PROJECT_ROOT / "models" / "top20_lgbm" / "best_lgbm.txt"
TUNING_PATH = PROJECT_ROOT / "models" / "top20_lgbm" / "tuning_results.csv"

TARGET_COL = "responder_6"
WEIGHT_COL = "weight"
DATE_COL = "date_id"
SYMBOL_COL = "symbol_id"
TIME_COL = "time_id"
CATEGORICAL_COLS = ["symbol_id", "feature_11"]

SAMPLE_ROWS_FOR_SCATTER = 100_000
RANDOM_STATE = 42

In [ ]:
def weighted_r2(y_true, y_pred, weight):
    y_true = np.asarray(y_true, dtype=np.float64)
    y_pred = np.asarray(y_pred, dtype=np.float64)
    weight = np.asarray(weight, dtype=np.float64)
    numerator = np.sum(weight * np.square(y_true - y_pred))
    denominator = np.sum(weight * np.square(y_true))
    return np.nan if denominator == 0 else 1.0 - numerator / denominator


def weighted_rmse(y_true, y_pred, weight):
    y_true = np.asarray(y_true, dtype=np.float64)
    y_pred = np.asarray(y_pred, dtype=np.float64)
    weight = np.asarray(weight, dtype=np.float64)
    return np.sqrt(np.average(np.square(y_true - y_pred), weights=weight))


def load_split(name):
    path = DATA_DIR / f"{name}.parquet"
    if not path.exists():
        raise FileNotFoundError(path)
    df = pd.read_parquet(path)
    for col in CATEGORICAL_COLS:
        if col in df.columns:
            df[col] = df[col].astype("category")
    return df


def add_predictions(df, model, feature_cols):
    out = df.copy()
    out["prediction"] = model.predict(out[feature_cols], num_iteration=model.best_iteration)
    out["residual"] = out[TARGET_COL] - out["prediction"]
    out["abs_residual"] = out["residual"].abs()
    return out

In [ ]:
if TUNING_PATH.exists():
    display(pd.read_csv(TUNING_PATH))

model = lgb.Booster(model_file=str(MODEL_PATH))
train_df = load_split("train")
valid_df = load_split("valid")
test_df = load_split("test")

feature_cols = [col for col in train_df.columns if col not in [DATE_COL, TARGET_COL, WEIGHT_COL]]
print({
    "train": train_df.shape,
    "valid": valid_df.shape,
    "test": test_df.shape,
    "features": len(feature_cols),
})

In [ ]:
train_pred = add_predictions(train_df, model, feature_cols)
valid_pred = add_predictions(valid_df, model, feature_cols)
test_pred = add_predictions(test_df, model, feature_cols)

metrics = []
for name, df in [("train", train_pred), ("valid", valid_pred), ("test", test_pred)]:
    metrics.append({
        "split": name,
        "rows": len(df),
        "date_min": int(df[DATE_COL].min()),
        "date_max": int(df[DATE_COL].max()),
        "weighted_r2": weighted_r2(df[TARGET_COL], df["prediction"], df[WEIGHT_COL]),
        "weighted_rmse": weighted_rmse(df[TARGET_COL], df["prediction"], df[WEIGHT_COL]),
        "pred_mean": df["prediction"].mean(),
        "pred_std": df["prediction"].std(),
        "target_mean": df[TARGET_COL].mean(),
        "target_std": df[TARGET_COL].std(),
    })

metrics_df = pd.DataFrame(metrics)
metrics_df

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

sns.barplot(data=metrics_df, x="split", y="weighted_r2", ax=axes[0])
axes[0].axhline(0, color="black", linewidth=1)
axes[0].set_title("Weighted R2 by Split")

sns.barplot(data=metrics_df, x="split", y="weighted_rmse", ax=axes[1])
axes[1].set_title("Weighted RMSE by Split")

plt.tight_layout()

In [ ]:
plot_df = test_pred.sample(min(SAMPLE_ROWS_FOR_SCATTER, len(test_pred)), random_state=RANDOM_STATE)

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

sns.histplot(plot_df[TARGET_COL], bins=80, stat="density", label="actual", ax=axes[0], color="steelblue", alpha=0.45)
sns.histplot(plot_df["prediction"], bins=80, stat="density", label="prediction", ax=axes[0], color="darkorange", alpha=0.45)
axes[0].set_title("Test Actual vs Predicted Distribution")
axes[0].legend()

sns.scatterplot(data=plot_df, x=TARGET_COL, y="prediction", s=8, alpha=0.25, ax=axes[1])
axes[1].axhline(0, color="black", linewidth=1)
axes[1].axvline(0, color="black", linewidth=1)
axes[1].set_title("Test Predictions vs Actuals")

plt.tight_layout()

In [ ]:
def symbol_metrics(df, split_name):
    rows = []
    for symbol_id, group in df.groupby(SYMBOL_COL, observed=True):
        rows.append({
            "split": split_name,
            "symbol_id": symbol_id,
            "rows": len(group),
            "weight_sum": group[WEIGHT_COL].sum(),
            "weighted_r2": weighted_r2(group[TARGET_COL], group["prediction"], group[WEIGHT_COL]),
            "pred_mean": group["prediction"].mean(),
            "target_mean": group[TARGET_COL].mean(),
            "abs_residual_mean": group["abs_residual"].mean(),
        })
    return pd.DataFrame(rows)


symbol_df = pd.concat([
    symbol_metrics(valid_pred, "valid"),
    symbol_metrics(test_pred, "test"),
], ignore_index=True)

symbol_df.sort_values(["split", "weight_sum"], ascending=[True, False]).head(20)

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(14, 8), sharex=True)

sns.barplot(data=symbol_df[symbol_df["split"] == "test"], x="symbol_id", y="weighted_r2", ax=axes[0], color="steelblue")
axes[0].axhline(0, color="black", linewidth=1)
axes[0].set_title("Test Weighted R2 by Symbol")

sns.barplot(data=symbol_df[symbol_df["split"] == "test"], x="symbol_id", y="weight_sum", ax=axes[1], color="gray")
axes[1].set_title("Test Weight by Symbol")

plt.tight_layout()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

symbol_means = test_pred.groupby(SYMBOL_COL, observed=True).agg(
    pred_mean=("prediction", "mean"),
    target_mean=(TARGET_COL, "mean"),
    rows=(TARGET_COL, "size"),
).reset_index()

sns.lineplot(data=symbol_means, x=SYMBOL_COL, y="pred_mean", marker="o", label="prediction", ax=axes[0])
sns.lineplot(data=symbol_means, x=SYMBOL_COL, y="target_mean", marker="o", label="actual", ax=axes[0])
axes[0].axhline(0, color="black", linewidth=1)
axes[0].set_title("Mean Return by Symbol on Test")

sns.scatterplot(data=test_pred.sample(min(100_000, len(test_pred)), random_state=RANDOM_STATE), x=SYMBOL_COL, y="prediction", alpha=0.15, s=8, ax=axes[1])
axes[1].axhline(0, color="black", linewidth=1)
axes[1].set_title("Predicted Returns by Symbol on Test")

plt.tight_layout()

In [ ]:
def date_metrics(df, split_name):
    rows = []
    for date_id, group in df.groupby(DATE_COL):
        rows.append({
            "split": split_name,
            "date_id": date_id,
            "rows": len(group),
            "weighted_r2": weighted_r2(group[TARGET_COL], group["prediction"], group[WEIGHT_COL]),
            "pred_mean": group["prediction"].mean(),
            "target_mean": group[TARGET_COL].mean(),
            "weighted_rmse": weighted_rmse(group[TARGET_COL], group["prediction"], group[WEIGHT_COL]),
        })
    return pd.DataFrame(rows)


date_df = pd.concat([
    date_metrics(train_pred, "train"),
    date_metrics(valid_pred, "valid"),
    date_metrics(test_pred, "test"),
], ignore_index=True)

date_df.tail(10)

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(14, 10), sharex=True)

sns.lineplot(data=date_df, x="date_id", y="weighted_r2", hue="split", ax=axes[0])
axes[0].axhline(0, color="black", linewidth=1)
axes[0].set_title("Weighted R2 Over Date")

sns.lineplot(data=date_df, x="date_id", y="weighted_rmse", hue="split", ax=axes[1])
axes[1].set_title("Weighted RMSE Over Date")

sns.lineplot(data=date_df, x="date_id", y="pred_mean", hue="split", ax=axes[2])
sns.lineplot(data=date_df, x="date_id", y="target_mean", hue="split", ax=axes[2], linestyle="--")
axes[2].axhline(0, color="black", linewidth=1)
axes[2].set_title("Mean Prediction and Mean Actual Over Date")

plt.tight_layout()

In [ ]:
time_df = test_pred.groupby(TIME_COL).agg(
    rows=(TARGET_COL, "size"),
    pred_mean=("prediction", "mean"),
    target_mean=(TARGET_COL, "mean"),
    abs_residual_mean=("abs_residual", "mean"),
).reset_index()

fig, axes = plt.subplots(2, 1, figsize=(14, 7), sharex=True)

sns.lineplot(data=time_df, x=TIME_COL, y="pred_mean", label="prediction", ax=axes[0])
sns.lineplot(data=time_df, x=TIME_COL, y="target_mean", label="actual", ax=axes[0])
axes[0].axhline(0, color="black", linewidth=1)
axes[0].set_title("Mean Return by Intraday Time ID on Test")

sns.lineplot(data=time_df, x=TIME_COL, y="abs_residual_mean", ax=axes[1], color="firebrick")
axes[1].set_title("Mean Absolute Residual by Intraday Time ID on Test")

plt.tight_layout()

In [ ]:
importance_path = PROJECT_ROOT / "models" / "top20_lgbm" / "best_feature_importance.csv"
if importance_path.exists():
    importance = pd.read_csv(importance_path)
else:
    importance = pd.DataFrame({
        "feature": model.feature_name(),
        "importance_gain": model.feature_importance(importance_type="gain"),
        "importance_split": model.feature_importance(importance_type="split"),
    }).sort_values("importance_gain", ascending=False)

plt.figure(figsize=(10, 6))
sns.barplot(data=importance.head(20), y="feature", x="importance_gain", color="steelblue")
plt.title("Best Model Feature Importance")
plt.tight_layout()

importance.head(20)

## Tuning Comparison

In [ ]:
tuning = pd.read_csv(TUNING_PATH).sort_values("valid_weighted_r2", ascending=False).reset_index(drop=True)
tuning["rank"] = np.arange(1, len(tuning) + 1)
cols = [
    "rank", "name", "valid_weighted_r2", "best_iteration", "elapsed_seconds",
    "num_leaves", "learning_rate", "min_data_in_leaf",
    "feature_fraction", "bagging_fraction", "lambda_l2",
]
tuning[cols]


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

sns.barplot(data=tuning, x="valid_weighted_r2", y="name", ax=axes[0], color="steelblue")
axes[0].axvline(0, color="black", linewidth=1)
axes[0].set_title("Validation Weighted R2 by Config")

sns.barplot(data=tuning, x="best_iteration", y="name", ax=axes[1], color="gray")
axes[1].set_title("Best Iteration by Config")

plt.tight_layout()


In [ ]:
best_config = tuning.iloc[0].to_dict()
second_config = tuning.iloc[1].to_dict() if len(tuning) > 1 else None

print(f"best config: {best_config['name']}")
print(f"best valid weighted R2: {best_config['valid_weighted_r2']:.8f}")
print(f"best iteration: {int(best_config['best_iteration'])}")

if second_config is not None:
    gap = best_config["valid_weighted_r2"] - second_config["valid_weighted_r2"]
    print(f"gap vs second best: {gap:.8f}")

print("test score from saved best model:")
display(metrics_df[metrics_df["split"] == "test"][["weighted_r2", "weighted_rmse", "pred_mean", "pred_std", "target_mean", "target_std"]])
